# Step D: evaluate all 15 models (Colab)

Runs `evaluation/evaluate_v2.py`: 5 models (M0, M1, M4, M5, M6) x seeds 0, 1, 2, on full 3D volumes under the 13 DEC-005 conditions.
The checkpoints are in the repo (`results/train_v3/`), so nothing but BraTS and the code needs to be provided.

**Order matters (brief `docs/handoff/TASK_NAZIR_STEP_D.md`):**
1. Set `SPLIT = 'val'`, *Run all*. Debug on the 37 validation patients and read the acceptance checks.
2. Only when every check passes: set `SPLIT = 'test'`, `CONFIRM_TEST = True`, run again. **The test set is run once**; nobody looks at test numbers before then.

Everything is written to Google Drive and the run is resumable: if Colab disconnects, *Run all* again and finished (patient, condition) pairs are skipped.
Runtime: *Change runtime type -> L4 GPU* (T4 also works, slower).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- Configuration ----
import os
SPLIT        = 'val'            # 'val' while debugging; 'test' for the one final run
CONFIRM_TEST = False            # must be True for SPLIT = 'test'
MAX_PATIENTS = 0                # 0 = all patients of the split; use 2 for a quick smoke test
ZIP    = '/content/drive/MyDrive/Argus/archive.zip'                  # BraTS 2020 zip, as in the training notebook
OUT    = f'/content/drive/MyDrive/Argus/eval_v2/{SPLIT}'             # CSV + predictions/ land here (resumable)
REPO, BRANCH = 'https://github.com/<your-github-user>/Argus.git', 'nazir-step-d'   # <-- fill in after pushing the branch
os.makedirs(OUT, exist_ok=True)

In [ ]:
if os.path.isdir('/content/Argus'):
    !git -C /content/Argus pull -q
else:
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/Argus
%cd /content/Argus
!pip -q install nibabel medpy
!ls results/train_v3/*/seed_*/best_model.pt | wc -l      # expect 15

In [ ]:
# Extract only the patients of this split (37 val or 74 test) plus name_mapping.csv
import json, zipfile, glob
need = json.load(open('data/splits/split_v1.json'))[SPLIT]
RAW = (glob.glob('/content/brats/**/MICCAI_BraTS2020_TrainingData', recursive=True) or [''])[0]
have = RAW and all(os.path.isdir(f'{RAW}/{p}') for p in need)
if not have:
    with zipfile.ZipFile(ZIP) as z:
        names = [n for n in z.namelist() if any(f'/{p}/' in n for p in need) or n.endswith('/name_mapping.csv')]
        z.extractall('/content/brats', members=names)
    RAW = glob.glob('/content/brats/**/MICCAI_BraTS2020_TrainingData', recursive=True)[0]
print(RAW, len(need), 'patients')

In [ ]:
# Unit tests (seconds) before spending GPU time
!python -m pytest tests/test_evaluate_v2.py -q

In [ ]:
# Evaluation. Prints seconds per (patient, condition) and the projected time of the full test run.
test_flag = '--confirm_test' if CONFIRM_TEST else ''
!python -m evaluation.evaluate_v2 --raw_dir {RAW} --split {SPLIT} {test_flag} --max_patients {MAX_PATIENTS} \
    --out_csv {OUT}/raw_metrics_v2.csv --out_dir {OUT}

## Acceptance checks (validation only)
Dice reproduces the training logs within 0.005, HD95 empty-mask convention, NIfTI round trip, timing printed above.

In [ ]:
if SPLIT == 'val':
    !python -m evaluation.check_val --raw_dir {RAW} --csv {OUT}/raw_metrics_v2.csv --pred_dir {OUT}/predictions
else:
    print('Acceptance checks are for the validation run only.')

In [ ]:
# Quick look at the numbers (validation: fine to inspect; test: only after the run is complete)
import pandas as pd
df = pd.read_csv(f'{OUT}/raw_metrics_v2.csv')
print(len(df), 'rows,', df.patient_id.nunique(), 'patients,', df.groupby(['model', 'seed']).ngroups, 'checkpoints')
df[df.condition_id.isin(['clean', 'kspace_snr5_r1.0_t1', 'kspace_snr3_r0.5_t5'])] \
  .groupby(['model', 'condition_id']).dice_mean.mean().unstack().round(4)

In [ ]:
# Test run only: zip the seed-0 predictions for Joshua's MATLAB pipeline (not for git)
if SPLIT == 'test':
    !cd {OUT} && zip -qr predictions_test_seed0.zip predictions && ls -lh predictions_test_seed0.zip
    print('Commit results/raw_metrics_v2.csv (copy it from', OUT, '); share the zip with Joshua. Do not commit predictions.')